In [31]:
import os, sys, time
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
pd.options.mode.chained_assignment = None
import h5py
import sqlalchemy
from shapely import wkt
import geopandas as gpd
import seaborn as sns
from itertools import cycle, islice
import pyodbc
import warnings
import matplotlib.pyplot as plt
import psrcelmerpy
import numpy as np
from scipy.spatial import cKDTree

In [32]:
df_lu = pd.read_csv(r'C:\Workspace\displacement_index\parcels_urbansim.txt',
                   delim_whitespace=True)

# Load as a geodataframe
gdf_lu = gpd.GeoDataFrame(
    df_lu, geometry=gpd.points_from_xy(df_lu.xcoord_p, df_lu.ycoord_p))

crs = {'init' : 'EPSG:2285'}
gdf_lu.crs = crs
base_year = "2023"

parcel_geog = pd.read_sql_table('parcel_'+base_year+'_geography', 'sqlite:///N:/rtp_2026_2050/final_runs/sc_base_year_2023_final/soundcast/inputs/db/soundcast_inputs_2023.db')

In [11]:
parcel_geog_query = pd.read_sql_query('SELECT ParcelID, geometry, GEOID20 FROM parcel_'+base_year+'_geography', 'sqlite:///N:/rtp_2026_2050/final_runs/sc_base_year_2023_final/soundcast/inputs/db/soundcast_inputs_2023.db')

In [12]:
parcel_geog_query

,ParcelID,geometry,GEOID20
0,1,POINT (1292255.14427 162728.617255),5.303303e+14
1,2,POINT (1291832.24123 164041.742835),5.303303e+14
2,3,POINT (1291594.61453 164048.669737),5.303303e+14
3,4,POINT (1291539.63503 164050.178628),5.303303e+14
4,5,POINT (1291479.35506 164042.397388),5.303303e+14
...,...,...,...
1329923,1329924,POINT (1444396.3 305434.297345),5.306105e+14
1329924,1329925,POINT (1310124.17381 305444.996829),5.306105e+14
1329925,1329926,POINT (1294709.29501 305556.196074),5.306105e+14
1329926,1329927,POINT (1306089.39348 288830.86527),5.306105e+14


In [33]:
gdf_lu['geometry']

0          POINT (1292255.144 162728.617)
1          POINT (1291832.241 164041.743)
2           POINT (1291594.615 164048.67)
3          POINT (1291539.635 164050.179)
4          POINT (1291479.355 164042.397)
                        ...              
1329923      POINT (1444396.3 305434.297)
1329924    POINT (1310124.174 305444.997)
1329925    POINT (1294709.295 305556.196)
1329926    POINT (1306089.393 288830.865)
1329927    POINT (1313226.373 286303.836)
Name: geometry, Length: 1329928, dtype: geometry

In [34]:
parcel_geog["ParcelID"]

0                1
1                2
2                3
3                4
4                5
            ...   
1329923    1329924
1329924    1329925
1329925    1329926
1329926    1329927
1329927    1329928
Name: ParcelID, Length: 1329928, dtype: int64

In [35]:
new_df_lu = gdf_lu.merge(parcel_geog,left_on='parcelid', right_on='ParcelID', how='left')


In [36]:
new_df_lu['geometry_x']

0          POINT (1292255.144 162728.617)
1          POINT (1291832.241 164041.743)
2           POINT (1291594.615 164048.67)
3          POINT (1291539.635 164050.179)
4          POINT (1291479.355 164042.397)
                        ...              
1329923      POINT (1444396.3 305434.297)
1329924    POINT (1310124.174 305444.997)
1329925    POINT (1294709.295 305556.196)
1329926    POINT (1306089.393 288830.865)
1329927    POINT (1313226.373 286303.836)
Name: geometry_x, Length: 1329928, dtype: geometry

In [ ]:
new_df_lu.columns

,aparks,empedu_p,empfoo_p,empgov_p,empind_p,empmed_p,empofc_p,empoth_p,empret_p,emprsc_p,...,equity_focus_areas_2023__efa_pov200,equity_focus_areas_2023__efa_lep,equity_focus_areas_2023__efa_youth,equity_focus_areas_2023__efa_older,equity_focus_areas_2023__efa_dis,BaseYear,all_day_transit,frequent_transit,hct,min_transit
0,0,0,0,0,0,0,0,0,0,0,...,2.0,2.0,0.0,0.0,2.0,2023,0.0,0.0,0.0,0.0
1,0,0,0,0,0,0,0,0,0,0,...,2.0,2.0,0.0,0.0,2.0,2023,1.0,0.0,0.0,1.0
2,0,0,0,0,0,0,0,1,0,0,...,2.0,2.0,0.0,0.0,2.0,2023,1.0,1.0,0.0,1.0
3,0,0,0,0,1,0,4,1,0,0,...,2.0,2.0,0.0,0.0,2.0,2023,1.0,1.0,0.0,1.0
4,0,0,0,0,0,0,0,0,0,0,...,2.0,2.0,0.0,0.0,2.0,2023,1.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1329923,0,0,0,0,0,0,0,0,0,0,...,1.0,0.0,0.0,1.0,1.0,2023,0.0,0.0,0.0,0.0
1329924,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,2.0,0.0,0.0,2023,0.0,0.0,0.0,0.0
1329925,0,0,0,0,0,0,0,0,0,0,...,0.0,1.0,1.0,0.0,0.0,2023,0.0,0.0,0.0,1.0
1329926,0,0,0,0,0,0,0,0,0,0,...,0.0,1.0,1.0,0.0,0.0,2023,0.0,0.0,0.0,0.0


In [37]:
eg_conn = psrcelmerpy.ElmerGeoConn()
parks_gdf = eg_conn.read_geolayer('open_space_parks').query("site_type == 'Park'")
# open_space_parks_gdf = eg_conn.read_geolayer('open_space_parks')
parks_gdf

,OBJECTID,site_name,site_type,owner,owner_type,acres,object_id,size,buffer,join_id,SDE_STATE_ID,source,Shape,geometry
0,10,Lost Lake,Park,Vashon PRSD,Park District,45.64343682,NaN,Community,1 miles,10.0,1352,Open Space Plan,POLYGON ((1230512.3302300572 134506.7399704754...,"POLYGON ((-122.48717 47.35667, -122.48704 47.3..."
1,618,First Hill Park - Seattle,Park,City of Seattle,City,0.21696463,NaN,Neighborhood,0.5 miles,618.0,1352,Open Space Plan,POLYGON ((1272449.8779164702 226773.6430347263...,"POLYGON ((-122.32523 47.61191, -122.32539 47.6..."
2,1069,Juanita Heights Park,Park,City of Kirkland,City,5.87811142,NaN,Neighborhood,0.5 miles,1069.0,1352,Open Space Plan,POLYGON ((1298483.5174501389 262506.8622505664...,"POLYGON ((-122.22231 47.71119, -122.22275 47.7..."
3,1520,Dorothy Bothell,Park,City of Auburn,City,4.31812082,NaN,Neighborhood,0.5 miles,1552.0,1352,Open Space Plan,"POLYGON ((1296976.8642954677 97018.5823353976,...","POLYGON ((-122.21635 47.25751, -122.21655 47.2..."
4,2128,Lord Hill Park,Park,SNOHOMISH COUNTY,County,1456.93890942,NaN,Regional,25 miles,2259.0,1352,Open Space Plan,MULTIPOLYGON (((1343113.0522066355 316570.2311...,"MULTIPOLYGON (((-122.04451 47.86145, -122.0445..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2858,623,Harborview Park,Park,Other King County,County,0.85907273,NaN,Neighborhood,0.5 miles,623.0,1352,Open Space Plan,POLYGON ((1272345.2829813063 223652.2890412211...,"POLYGON ((-122.32541 47.60335, -122.32498 47.6..."
2859,1074,Highlands Park - Kirkland,Park,City of Kirkland,City,2.45791294,NaN,Neighborhood,0.5 miles,1074.0,1352,Open Space Plan,POLYGON ((1306241.4799525589 255148.3290673941...,"POLYGON ((-122.19028 47.6914, -122.19052 47.69..."
2860,1682,Colegate Park,Park,City of University Place,City,11.84668131,1108.0,Neighborhood,0.5 miles,1746.0,1352,Open Space Plan,POLYGON ((1214400.2883072197 87285.78454381227...,"POLYGON ((-122.54791 47.22628, -122.54846 47.2..."
2861,2133,Cedar Grove Park,Park,CITY OF MONROE,City,0.50738503,NaN,Neighborhood,0.5 miles,2264.0,1352,Open Space Plan,MULTIPOLYGON (((1352585.1389054656 314489.0217...,"MULTIPOLYGON (((-122.0058 47.85615, -122.00547..."


In [38]:
park_centroids = parks_gdf.copy()
crs = {'init' : 'EPSG:2285'}
park_centroids = park_centroids.to_crs(crs)
park_centroids['geometry'] = park_centroids.geometry.centroid    
# park_centroids.crs

In [39]:
h5_file = h5py.File(r'N:\rtp_2026_2050\final_runs\sc_base_year_2023_final\soundcast\inputs\scenario\landuse\hh_and_persons.h5', 'r')
hh_df = pd.DataFrame()
h5_file['Household']['hhtaz'][0]
for col in h5_file['Household'].keys():
    hh_df[col] = h5_file['Household'][col][:]
h5_file.close()

In [40]:
df = hh_df[['hhparcel','hhsize']].groupby('hhparcel').sum()
df.rename(columns={'hhsize':'population'}, inplace=True)
print(df)

          population
hhparcel            
21                 7
27                92
70                88
71                 6
83                 2
...              ...
1327519            3
1327520            2
1327524            8
1327540            7
1327542            2

[1054131 rows x 1 columns]


In [12]:
park_centroids

,OBJECTID,site_name,site_type,owner,owner_type,acres,object_id,size,buffer,join_id,SDE_STATE_ID,source,Shape,geometry
0,10,Lost Lake,Park,Vashon PRSD,Park District,45.64343682,NaN,Community,1 miles,10.0,1352,Open Space Plan,POLYGON ((1230512.3302300572 134506.7399704754...,POINT (1229858.606 135074.471)
1,618,First Hill Park - Seattle,Park,City of Seattle,City,0.21696463,NaN,Neighborhood,0.5 miles,618.0,1352,Open Space Plan,POLYGON ((1272449.8779164702 226773.6430347263...,POINT (1272379.508 226773.388)
2,1069,Juanita Heights Park,Park,City of Kirkland,City,5.87811142,NaN,Neighborhood,0.5 miles,1069.0,1352,Open Space Plan,POLYGON ((1298483.5174501389 262506.8622505664...,POINT (1298629.336 262387.271)
3,1520,Dorothy Bothell,Park,City of Auburn,City,4.31812082,NaN,Neighborhood,0.5 miles,1552.0,1352,Open Space Plan,"POLYGON ((1296976.8642954677 97018.5823353976,...",POINT (1296891.552 96677.621)
4,2128,Lord Hill Park,Park,SNOHOMISH COUNTY,County,1456.93890942,NaN,Regional,25 miles,2259.0,1352,Open Space Plan,MULTIPOLYGON (((1343113.0522066355 316570.2311...,POINT (1342171.961 311589.359)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2858,623,Harborview Park,Park,Other King County,County,0.85907273,NaN,Neighborhood,0.5 miles,623.0,1352,Open Space Plan,POLYGON ((1272345.2829813063 223652.2890412211...,POINT (1272319.746 223815.24)
2859,1074,Highlands Park - Kirkland,Park,City of Kirkland,City,2.45791294,NaN,Neighborhood,0.5 miles,1074.0,1352,Open Space Plan,POLYGON ((1306241.4799525589 255148.3290673941...,POINT (1306343.72 254983.717)
2860,1682,Colegate Park,Park,City of University Place,City,11.84668131,1108.0,Neighborhood,0.5 miles,1746.0,1352,Open Space Plan,POLYGON ((1214400.2883072197 87285.78454381227...,POINT (1214209.279 86606.581)
2861,2133,Cedar Grove Park,Park,CITY OF MONROE,City,0.50738503,NaN,Neighborhood,0.5 miles,2264.0,1352,Open Space Plan,MULTIPOLYGON (((1352585.1389054656 314489.0217...,POINT (1352631.392 314542.093)


In [41]:
new_df_lu = new_df_lu.copy()
new_df_lu = new_df_lu.rename(columns={'geometry_x': 'geometry'})


In [42]:
new_df_lu = new_df_lu.merge(df, left_on='parcelid', right_on='hhparcel', how='inner')

In [43]:
new_df_lu = new_df_lu[["Census2020Tract", "parcelid", "geometry", "population"]]

In [44]:
len(new_df_lu)

1054131

In [24]:
park_centroids.crs

<Projected CRS: EPSG:2285>
Name: NAD83 / Washington North (ftUS)
Axis Info [cartesian]:
- E[east]: Easting (US survey foot)
- N[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - Washington - counties of Chelan; Clallam; Douglas; Ferry; Grant north of approximately 47°30'N; Island; Jefferson; King; Kitsap; Lincoln; Okanogan; Pend Oreille; San Juan; Skagit; Snohomish; Spokane; Stevens; Whatcom.
- bounds: (-124.79, 47.08, -117.02, 49.05)
Coordinate Operation:
- name: SPCS83 Washington North zone (US survey foot)
- method: Lambert Conic Conformal (2SP)
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [ ]:
# park_centroids = park_centroids.set_crs(4326, allow_override=True)
# park_centroids = park_centroids.to_crs(2285)

In [25]:
park_centroids.total_bounds

array([1103468.79355257,  -84403.38295804, 1513722.35050298,
        474263.57052938])

In [45]:
def find_nearest(gdA, gdB):
    """ Find nearest value between two geodataframes.
        Returns "dist" for distance between nearest points.
    """

    nA = np.array(list(gdA.geometry.apply(lambda x: (x.x, x.y))))
    nB = np.array(list(gdB.geometry.apply(lambda x: (x.x, x.y))))
    btree = cKDTree(nB)
    dist, idx = btree.query(nA, k=1)
    gdB_nearest = gdB.iloc[idx].drop(columns="geometry").reset_index(drop=True)
    gdf = pd.concat(
        [
            gdA.reset_index(drop=True),
            gdB_nearest,
            pd.Series(dist, name='dist')
        ], 
        axis=1)

    return gdf

In [46]:
def weighted_avg(df, val_col, wt_col, agg_col):
    """ Returns weighted average for specified aggregation. 
        
        Parameters
    ----------
    df : Pandas DataFrame 
    val_col: column name of the value being averaged
    wt_col: weight column name
    agg_col: column to be used for aggregation
    ----------
    """
    df = df.copy()
    df['wt_tot'] = df[val_col] * df[wt_col]
    
    # Aggregate sums for each group
    df_agg = df.groupby(agg_col)[[wt_col, 'wt_tot']].sum()
    
    # Compute weighted average
    df_agg['wt_avg'] = df_agg['wt_tot'] / df_agg[wt_col]
    
    return df_agg
    # df['wt_tot'] = df[val_col]*df[wt_col]
    # df_agg = df.groupby(agg_col).sum()
    # df_agg['wt_avg'] = df_agg['wt_tot']/df_agg[wt_col]
    
    # return df_agg[['wt_avg']]

In [47]:
print('Calculating weighted distance...')

# Calculate nearest stop for all transit stations at once
if len(park_centroids) == 0:
    raise ValueError("No parks available in park_centroids")

# Find nearest park for each parcel
nearest_df = find_nearest(new_df_lu, park_centroids)
nearest_df['miles'] = nearest_df['dist'] / 5280.0  # convert feet to miles

nearest_df.head()
# Aggregate parcel-level distances to population-weighted average at the tract level
tract_output_df = weighted_avg(nearest_df, val_col='miles', wt_col='population',
                                agg_col='Census2020Tract').reset_index()

tract_output_df

Calculating weighted distance...


,Census2020Tract,population,wt_tot,wt_avg
0,5.303300e+10,3670,468.306784,0.127604
1,5.303300e+10,4270,993.513668,0.232673
2,5.303300e+10,4401,1230.236476,0.279536
3,5.303300e+10,3989,1252.293674,0.313937
4,5.303300e+10,2831,1245.473932,0.439941
...,...,...,...,...
914,5.306105e+10,3780,8080.789824,2.137775
915,5.306105e+10,7707,4937.905030,0.640704
916,5.306105e+10,5807,5188.707960,0.893526
917,5.306194e+10,6595,13105.106654,1.987128


In [48]:
tract_output_df = tract_output_df.rename(columns={'wt_avg': 'avg_miles_to_park'})

In [50]:
tract_output_df.to_csv(r'C:\Workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\distance_to_parks_output_population.csv', index=False)

In [152]:
print(new_df_lu.total_bounds)
print(park_centroids.total_bounds)

[1099777.18725    -93023.8403113 1610615.20787    476785.443993 ]
[-123.00715834   46.76212174 -121.34797013   48.29206862]


In [144]:
nearest_df["Census2020Tract"].nunique()

923

In [157]:
tract_output_df.to_csv(r'C:\Workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\distance_to_parks_output.csv', index=False)

In [66]:
gdf = find_nearest(gdf_lu, park_centroids)
result = gdf[["xcoord_p", "ycoord_p", "geometry", "site_name", "site_type", "dist"]]
# gdf.head(100).to_csv(r'C:\workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\distance_to_parks.csv', index=False)


In [67]:
result.to_csv(r'C:\workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\distance_to_parks_minimize_cols.csv', index=False)

In [61]:
gdf.head(100).to_csv(r'C:\workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\distance_to_parks.csv', index=False)
